# MyPortfolioManagement - Complete Tutorial

This notebook demonstrates all major functions in the myPortfolioManagement library.

**Author:** Ferhat  
**Date:** December 2024

## Table of Contents
1. [Setup & Configuration](#setup)
2. [Data Fetching](#data-fetching)
3. [Returns Calculation](#returns-calculation)
4. [Utility Functions](#utility-functions)
5. [Portfolio Optimization](#portfolio-optimization)
6. [Performance Metrics](#performance-metrics)
7. [Low-Level Risk Metrics](#risk-metrics)
8. [Backtesting & Simulation](#backtesting)
9. [Bootstrapping](#bootstrapping)
10. [Clustering & Regime Detection](#clustering)
11. [Visualization](#plotting)
12. [Portfolio Selection](#portfolio-selection)
13. [Complete Workflow Example](#workflow)

## 1. Setup & Configuration <a id='setup'></a>

Import all necessary libraries and configure display settings.

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Fix for quantstats-lumi compatibility with newer Jupyter
import sys
if 'IPython' in sys.modules:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        # Monkey-patch to avoid the deprecated magic() call
        if not hasattr(ipython, 'magic'):
            ipython.magic = lambda x: None

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import quantstats_lumi as qs

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

# Matplotlib settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import quantstats_lumi as qs

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

# Matplotlib settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### Helper Function: Fix for Benchmark Returns

This fixes a quantstats DataFrame compatibility issue.

In [ ]:
def get_benchmark_returns_fixed(ticker: str = '^GSPC', name: str = 'S&P500') -> pd.DataFrame:
    """
    Fixed version of get_benchmark_returns that handles quantstats DataFrame output.
    
    Args:
        ticker: Yahoo Finance ticker symbol (default: ^GSPC for S&P 500)
        name: Name for the return series
    
    Returns:
        DataFrame with benchmark returns
    """
    ret_data = qs.utils.download_returns(ticker)
    
    # Handle both Series and DataFrame returns
    if isinstance(ret_data, pd.DataFrame):
        ret = ret_data.squeeze()
    else:
        ret = ret_data
    
    ret.name = name
    return pd.DataFrame(ret)

print("Helper function loaded!")

## 2. Data Fetching <a id='data-fetching'></a>

Fetch historical price data from Yahoo Finance for a diversified portfolio of ETFs.

In [ ]:
from myPortfolioManagement.myData import (
    get_stock_prices,
    get_stock_info,
    get_option_exp_dates
)

# Define a diversified portfolio of ETFs
tickers = [
    'SPY',   # S&P 500
    'QQQ',   # Nasdaq 100
    'IWM',   # Russell 2000 (Small Cap)
    'EFA',   # International Developed
    'EEM',   # Emerging Markets
    'AGG',   # US Bonds
    'TIP',   # TIPS (Inflation Protected)
    'GLD',   # Gold
    'VNQ',   # Real Estate
    'DBC',   # Commodities
]

print(f"Fetching data for {len(tickers)} tickers...")
print(f"Tickers: {', '.join(tickers)}")

In [ ]:
# Fetch historical prices
prices = get_stock_prices(
    yahoo_tickers=tickers,
    start_date='2018-01-01',
    end_date='2025-12-21',
    freq='daily',
    wide_format=True
)

print(f"\nPrice data shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
print("\nFirst 5 rows:")
prices.head()

In [ ]:
# Get stock information
stock_info = get_stock_info(tickers)
print("Stock Information:")
stock_info[['yahooTicker', 'longName', 'type', 'currency', 'sector']]

In [ ]:
# Plot normalized prices
normalized = prices / prices.iloc[0] * 100
fig, ax = plt.subplots(figsize=(14, 7))
normalized.plot(ax=ax)
ax.set_title('Normalized Prices (Base = 100)', fontsize=14)
ax.set_ylabel('Price')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Returns Calculation <a id='returns-calculation'></a>

Calculate returns in various formats and frequencies.

In [ ]:
from myPortfolioManagement.myReturns import (
    calculate_returns,
    average_returns,
    convert_returns_freq,
    calculate_portfolio_returns
)

In [ ]:
# Daily returns
returns_daily = calculate_returns(prices, log_returns=False)
print(f"Daily returns shape: {returns_daily.shape}")
print("\nDaily returns statistics:")
returns_daily.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# Log returns
returns_log = calculate_returns(prices, log_returns=True)
print("Log returns statistics:")
returns_log.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# Monthly returns
returns_monthly = calculate_returns(prices, convert_to='monthly')
print(f"Monthly returns shape: {returns_monthly.shape}")
print("\nLast 12 months:")
returns_monthly.tail(12)

In [ ]:
# Expected returns (different methods)
mu_hist = average_returns(returns_daily, method='hist', periods=252)
mu_ema = average_returns(returns_daily, method='ema', span=500, periods=252)

expected_returns = pd.DataFrame({
    'Historical': mu_hist,
    'EMA': mu_ema
}).sort_values('Historical', ascending=False)

print("Expected Annual Returns:")
expected_returns

In [ ]:
# Get benchmark returns
benchmark_sp500 = get_benchmark_returns_fixed('^GSPC', 'S&P500')
print(f"S&P 500 benchmark shape: {benchmark_sp500.shape}")
print("\nLast 5 days:")
benchmark_sp500.tail()

## 4. Utility Functions <a id='utility-functions'></a>

Data quality checks and helper functions.

In [ ]:
from myPortfolioManagement.myUtils import (
    data_overview,
    balance_dates,
    check_date_index,
    cap_outliersTS
)

In [ ]:
# Data overview (convert to long format first)
prices_long = prices.reset_index().melt(
    id_vars='Date', 
    var_name='asset', 
    value_name='price'
)

overview = data_overview(
    prices_long,
    my_assets_col_name='asset',
    my_date_col_name='Date',
    price_col_name='price'
)

print("Data Overview:")
overview

In [ ]:
# Balance dates between returns and benchmark
returns_balanced, benchmark_balanced = balance_dates(returns_daily, benchmark_sp500)
print(f"Balanced returns shape: {returns_balanced.shape}")
print(f"Balanced benchmark shape: {benchmark_balanced.shape}")

## 5. Portfolio Optimization <a id='portfolio-optimization'></a>

Generate optimal portfolios using various methods.

In [ ]:
from myPortfolioManagement.myPortfolioOptimisation import (
    HRP,
    equal_weight_portfolio,
    inverse_vol_portfolio,
    port_GMV,
    port_max_sharpe,
    port_target_return,
    port_target_volatility,
    port_CVAR,
    generate_rp_portfolios,
    make_standard_portfolios,
    risk_contributions
)

# Prepare returns for optimization (fill NaN with 0)
returns_opt = returns_daily.fillna(0)
print("Ready for optimization!")

In [ ]:
# Equal Weight Portfolio
weights_equal = equal_weight_portfolio(returns_opt)
print("Equal Weight Portfolio:")
weights_equal

In [ ]:
# Inverse Volatility Portfolio
weights_inv_vol = inverse_vol_portfolio(returns_opt, weight_max=0.25)
print("Inverse Volatility Portfolio:")
weights_inv_vol.sort_values('port_inverse_vol', ascending=False)

In [ ]:
# Hierarchical Risk Parity (HRP)
weights_hrp = HRP(
    model='HRP',
    returns_training=returns_opt,
    codependence='pearson',
    covariance='ledoit',
    rm='MV',
    linkage='ward',
    weight_max=0.25,
    weight_min=0.02
)
print("HRP Portfolio:")
weights_hrp.sort_values('port_weight', ascending=False)

In [ ]:
# HERC (Hierarchical Equal Risk Contribution)
weights_herc = HRP(
    model='HERC',
    returns_training=returns_opt,
    codependence='pearson',
    rm='CVaR',
    weight_max=0.30,
    weight_min=0.02
)
print("HERC Portfolio:")
weights_herc.sort_values('port_weight', ascending=False)

In [ ]:
# Global Minimum Variance
weights_gmv = port_GMV(returns_opt, weight_min=0.02, weight_max=0.30)
print("Global Minimum Variance Portfolio:")
weights_gmv.sort_values('port_min_vol', ascending=False)

In [ ]:
# Maximum Sharpe Ratio
weights_max_sharpe = port_max_sharpe(returns_opt, rf=0.04, weight_min=0.02, weight_max=0.30)
print("Max Sharpe Portfolio:")
weights_max_sharpe.sort_values('port_max_Sharpe', ascending=False)

In [ ]:
# Minimum CVaR Portfolio
weights_cvar = port_CVAR(returns_opt, confidence_interval=0.95, rf=0.04, 
                         weight_min=0.02, weight_max=0.30)
print("Minimum CVaR Portfolio:")
weights_cvar.sort_values('port_target_CVAR', ascending=False)

In [ ]:
# Compare all portfolios
# First, let's inspect the structure of each weights dataframe
print("Inspecting weight dataframes...")
print("\nweights_equal columns:", weights_equal.columns.tolist())
print("weights_inv_vol columns:", weights_inv_vol.columns.tolist())
print("weights_hrp columns:", weights_hrp.columns.tolist())
print("weights_herc columns:", weights_herc.columns.tolist())
print("weights_gmv columns:", weights_gmv.columns.tolist())
print("weights_max_sharpe columns:", weights_max_sharpe.columns.tolist())
print("weights_cvar columns:", weights_cvar.columns.tolist())

# Now build the comparison dataframe more robustly
all_weights = pd.DataFrame(index=weights_hrp.index)

# Extract weights, handling different column names
all_weights['Equal'] = weights_equal['port_naive'].values if 'port_naive' in weights_equal.columns else weights_equal.iloc[:, -1].values
all_weights['Inv_Vol'] = weights_inv_vol['port_inverse_vol'].values if 'port_inverse_vol' in weights_inv_vol.columns else weights_inv_vol.iloc[:, -1].values
all_weights['HRP'] = weights_hrp['port_weight'].values if 'port_weight' in weights_hrp.columns else weights_hrp.iloc[:, -1].values
all_weights['HERC'] = weights_herc['port_weight'].values if 'port_weight' in weights_herc.columns else weights_herc.iloc[:, -1].values

# For GMV, Max_Sharpe, and CVaR - handle cases where 'asset' might be index or column
if 'asset' in weights_gmv.columns:
    all_weights['GMV'] = weights_gmv.set_index('asset')['port_min_vol']
else:
    all_weights['GMV'] = weights_gmv['port_min_vol'].values if 'port_min_vol' in weights_gmv.columns else weights_gmv.iloc[:, -1].values

if 'asset' in weights_max_sharpe.columns:
    all_weights['Max_Sharpe'] = weights_max_sharpe.set_index('asset')['port_max_Sharpe']
else:
    all_weights['Max_Sharpe'] = weights_max_sharpe['port_max_Sharpe'].values if 'port_max_Sharpe' in weights_max_sharpe.columns else weights_max_sharpe.iloc[:, -1].values

if 'asset' in weights_cvar.columns:
    all_weights['Min_CVaR'] = weights_cvar.set_index('asset')['port_target_CVAR']
else:
    all_weights['Min_CVaR'] = weights_cvar['port_target_CVAR'].values if 'port_target_CVAR' in weights_cvar.columns else weights_cvar.iloc[:, -1].values

print("\nAll Portfolio Weights Comparison:")
all_weights.round(3)

In [ ]:
# Visualize weights comparison
fig, ax = plt.subplots(figsize=(14, 6))
all_weights.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Portfolio Weights Comparison', fontsize=14)
ax.set_xlabel('Asset')
ax.set_ylabel('Weight')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate portfolio returns using optimized weights
print("Calculating portfolio returns using optimized weights...\n")

# Create a dictionary to store portfolio returns
portfolio_returns_dict = {}

# HRP Portfolio
hrp_weights = weights_hrp['port_weight'].to_dict()
portfolio_returns_dict['HRP'] = (returns_daily * pd.Series(hrp_weights)).sum(axis=1)

# HERC Portfolio
herc_weights = weights_herc['port_weight'].to_dict()
portfolio_returns_dict['HERC'] = (returns_daily * pd.Series(herc_weights)).sum(axis=1)

# Equal Weight Portfolio
equal_weights = weights_equal['port_naive'].to_dict()
portfolio_returns_dict['Equal_Weight'] = (returns_daily * pd.Series(equal_weights)).sum(axis=1)

# Inverse Volatility Portfolio
inv_vol_weights = weights_inv_vol['port_inverse_vol'].to_dict()
portfolio_returns_dict['Inv_Vol'] = (returns_daily * pd.Series(inv_vol_weights)).sum(axis=1)

# GMV Portfolio
gmv_weights = weights_gmv['port_min_vol'].to_dict()
portfolio_returns_dict['GMV'] = (returns_daily * pd.Series(gmv_weights)).sum(axis=1)

# Max Sharpe Portfolio
max_sharpe_weights = weights_max_sharpe['port_max_Sharpe'].to_dict()
portfolio_returns_dict['Max_Sharpe'] = (returns_daily * pd.Series(max_sharpe_weights)).sum(axis=1)

# Min CVaR Portfolio
cvar_weights = weights_cvar['port_target_CVAR'].to_dict()
portfolio_returns_dict['Min_CVaR'] = (returns_daily * pd.Series(cvar_weights)).sum(axis=1)

# Combine into DataFrame
portfolio_returns_comparison = pd.DataFrame(portfolio_returns_dict)

# Calculate cumulative returns (growth of $1)
cum_portfolio_returns = (1 + portfolio_returns_comparison).cumprod()

print("Portfolio Cumulative Performance Summary:")
print(cum_portfolio_returns.iloc[-1].round(4))

# Recalculate max drawdown properly
max_drawdowns = {}
for col in portfolio_returns_comparison.columns:
    cum_ret = (1 + portfolio_returns_comparison[col]).cumprod()
    running_max = cum_ret.expanding().max()
    drawdown = (cum_ret - running_max) / running_max
    max_drawdowns[col] = drawdown.min()

print("\nPerformance Metrics:")
metrics_df = pd.DataFrame({
    'Annual Return': portfolio_returns_comparison.mean() * 252,
    'Annual Volatility': portfolio_returns_comparison.std() * np.sqrt(252),
    'Sharpe Ratio': (portfolio_returns_comparison.mean() * 252) / (portfolio_returns_comparison.std() * np.sqrt(252)),
    'Max Drawdown': pd.Series(max_drawdowns)
})
print(metrics_df.round(4))

In [ ]:
# Visualize cumulative returns for all portfolios vs S&P 500 benchmark
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Align benchmark returns with portfolio returns
benchmark_aligned = benchmark_sp500.squeeze().loc[portfolio_returns_comparison.index]
benchmark_cum = (1 + benchmark_aligned).cumprod()

# Plot 1: Cumulative Returns vs Benchmark
ax = axes[0, 0]
cum_portfolio_returns.plot(ax=ax, linewidth=2, label='Portfolios')
benchmark_cum.plot(ax=ax, linewidth=2.5, linestyle='--', color='black', label='S&P 500 Benchmark', alpha=0.8)
ax.set_title('Portfolio Cumulative Returns vs S&P 500', fontsize=12, fontweight='bold')
ax.set_ylabel('Growth of $1')
ax.set_xlabel('Date')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 2: Drawdown Comparison (Portfolios + Benchmark)
ax = axes[0, 1]
drawdowns = {}
for col in portfolio_returns_comparison.columns:
    cum_ret = (1 + portfolio_returns_comparison[col]).cumprod()
    running_max = cum_ret.expanding().max()
    drawdowns[col] = (cum_ret - running_max) / running_max

# Add benchmark drawdown
bench_running_max = benchmark_cum.expanding().max()
drawdowns['S&P 500'] = (benchmark_cum - bench_running_max) / bench_running_max

drawdowns_df = pd.DataFrame(drawdowns)
drawdowns_df.plot(ax=ax, linewidth=2)
ax.set_title('Drawdown Over Time (vs S&P 500)', fontsize=12, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 3: Rolling Volatility (252-day window) vs Benchmark
ax = axes[1, 0]
rolling_vol = portfolio_returns_comparison.rolling(252).std() * np.sqrt(252)
bench_rolling_vol = benchmark_aligned.rolling(252).std() * np.sqrt(252)

rolling_vol.plot(ax=ax, linewidth=2, label='Portfolios')
bench_rolling_vol.plot(ax=ax, linewidth=2.5, linestyle='--', color='black', label='S&P 500', alpha=0.8)
ax.set_title('Rolling Annual Volatility (252-day vs S&P 500)', fontsize=12, fontweight='bold')
ax.set_ylabel('Volatility')
ax.set_xlabel('Date')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 4: Performance Metrics Comparison (Including S&P 500)
ax = axes[1, 1]
benchmark_metrics = pd.DataFrame({
    'Annual Return': [benchmark_aligned.mean() * 252],
    'Annual Volatility': [benchmark_aligned.std() * np.sqrt(252)],
    'Sharpe Ratio': [(benchmark_aligned.mean() * 252) / (benchmark_aligned.std() * np.sqrt(252))],
    'Max Drawdown': [drawdowns['S&P 500'].min()]
}, index=['S&P 500'])

metrics_comparison = pd.concat([metrics_df, benchmark_metrics])
metrics_comparison['Annual Return'].plot(kind='barh', ax=ax, color=['steelblue']*len(metrics_df) + ['red'], alpha=0.7)
ax.set_title('Annual Return by Strategy vs S&P 500', fontsize=12, fontweight='bold')
ax.set_xlabel('Annual Return')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nVisualization complete!")
print("\nBenchmark (S&P 500) Metrics:")
print(benchmark_metrics.round(4))

## 6. Performance Metrics <a id='performance-metrics'></a>

Calculate comprehensive performance metrics.

In [ ]:
from myPortfolioManagement.myPerformanceMetrics import (
    get_main_stats,
    alpha_beta_table,
    alpha_beta_bull,
    alpha_beta_bear,
    information_ratio,
    drawdown_details,
    assets_drawdown_details,
    performance_overview,
    cagr,
    beta_Co_Moments_table
)

In [ ]:
# Main performance statistics
main_stats = get_main_stats(returns_daily, rf=0.04, smart=True)
print("Main Performance Stats:")
main_stats.round(4)

In [ ]:
# CAGR Calculation
cagr_values = cagr(prices)
print("Compound Annual Growth Rate:")
cagr_values.sort_values(ascending=False).round(4)

In [ ]:
# Alpha-Beta Analysis (Full Market)
ab_table = alpha_beta_table(returns_daily, benchmark_sp500, rf=0.04)
print("Alpha-Beta Table (Full, Bull, Bear):")
ab_table

In [ ]:
# Information Ratio
info_ratio = information_ratio(returns_daily, benchmark_sp500.squeeze())
print("Information Ratio:")
info_ratio.round(4)

In [ ]:
# Drawdown Details (for first asset)
spy_prices = prices.iloc[:, 0]
dd_details = drawdown_details(spy_prices, top_drawdowns=5)
print(f"Top 5 Drawdowns for {prices.columns[0]}:")
dd_details

In [ ]:
# Performance Overview
perf_overview = performance_overview(returns_daily, prices=False, short=True)
print("Performance Overview:")
perf_overview.round(4)

## 7. Low-Level Risk Metrics <a id='risk-metrics'></a>

Calculate individual risk and return metrics.

In [ ]:
from myPortfolioManagement.metrics import (
    vol, beta, var, cvar, lpm, hpm,
    max_dd, dd, sharpe_ratio, sortino_ratio,
    treynor_ratio, calmar_ratio, omega_ratio,
    gain_loss_ratio, upside_potential_ratio
)

# Use SPY returns for demonstration
spy_returns = returns_daily.iloc[:, 0].dropna().values
market_returns = benchmark_sp500.squeeze().dropna().values

# Align lengths
min_len = min(len(spy_returns), len(market_returns))
spy_returns = spy_returns[:min_len]
market_returns = market_returns[:min_len]

In [ ]:
# Basic risk metrics
print("Basic Risk Metrics:")
print(f"Volatility (annualized): {vol(spy_returns) * np.sqrt(252):.4f}")
print(f"Beta to market: {beta(spy_returns, market_returns):.4f}")
print(f"VaR (5%): {var(spy_returns, 0.05):.4f}")
print(f"CVaR (5%): {cvar(spy_returns, 0.05):.4f}")
print(f"Maximum Drawdown: {max_dd(spy_returns):.4f}")

In [ ]:
# Risk-adjusted return ratios
er = np.mean(spy_returns) * 252  # Annualized expected return
rf = 0.04  # Risk-free rate

print("Risk-Adjusted Return Ratios:")
print(f"Sharpe Ratio: {sharpe_ratio(er, spy_returns, rf/252):.4f}")
print(f"Sortino Ratio: {sortino_ratio(er, spy_returns, rf/252):.4f}")
print(f"Treynor Ratio: {treynor_ratio(er, spy_returns, market_returns, rf/252):.4f}")
print(f"Calmar Ratio: {calmar_ratio(er, spy_returns, rf/252):.4f}")
print(f"Omega Ratio: {omega_ratio(er, spy_returns, rf/252):.4f}")
print(f"Gain/Loss Ratio: {gain_loss_ratio(spy_returns):.4f}")

## 8. Backtesting & Simulation <a id='backtesting'></a>

Monte Carlo simulation and portfolio backtesting.

In [ ]:
from myPortfolioManagement.myBacktesting import (
    bootstrap_stats,
    bootstrap_portfolio_performance,
    sim_series,
    beating_probability,
    performance
)

# Create a simple portfolio return series
portfolio_returns = returns_daily.mean(axis=1)
portfolio_returns.name = 'Portfolio'

In [ ]:
# Bootstrap statistics
print("Running bootstrap statistics (500 simulations)...")
bootstrap_results = bootstrap_stats(
    returns=portfolio_returns,
    returns_benchmark=benchmark_sp500.squeeze(),
    rf=0.04,
    periods=252,
    n_sim=500
)
print("\nBootstrap Statistics Distribution:")
bootstrap_results.describe().round(4)

In [ ]:
# Bootstrap portfolio performance (in-sample vs out-of-sample)
print("Running bootstrap performance analysis...")
try:
    means, distributions, dist_stats = bootstrap_portfolio_performance(
        returns=portfolio_returns,
        returns_benchmark=benchmark_sp500.squeeze(),
        periods=252,
        rf=0.04,
        out_of_sample_date='2023-01-01',
        n_sim=500
    )
    print("\nMean Performance Metrics:")
    print(means.round(4))
except Exception as e:
    print(f"Bootstrap performance analysis: {e}")

In [ ]:
# Probability of beating benchmark
print("Calculating beating probability...")
try:
    beat_prob = beating_probability(
        returns=pd.DataFrame(portfolio_returns),
        returns_benchmark=benchmark_sp500,
        n_sample=500
    )
    print(f"\nProbability of beating S&P 500: {beat_prob.values[0][0]:.2%}")
except Exception as e:
    print(f"Beating probability calculation: {e}")

## 9. Bootstrapping <a id='bootstrapping'></a>

Time series bootstrapping methods.

In [ ]:
from myPortfolioManagement.myBootstrapping import (
    BootstrapIDD,
    BootstrapStationary,
    BootstrapCircular,
    BootstrapMovingBlock,
    bootstrappingTS
)

# Use single asset returns for bootstrapping
single_returns = returns_daily.iloc[:, 0].dropna()

In [ ]:
# IID Bootstrap
bs_iid = BootstrapIDD(series=single_returns, n_samples=100, seed=42)
print(f"IID Bootstrap shape: {bs_iid.shape}")
print(f"Original std: {single_returns.std():.6f}")
print(f"Bootstrap mean std: {bs_iid.std().mean():.6f}")

In [ ]:
# Stationary Bootstrap
bs_stationary = BootstrapStationary(
    series=single_returns,
    block_size=20,
    n_samples=100,
    seed=42
)
print(f"Stationary Bootstrap shape: {bs_stationary.shape}")

In [ ]:
# Circular Block Bootstrap
bs_circular = BootstrapCircular(
    series=single_returns,
    block_size=20,
    n_samples=100,
    seed=42
)
print(f"Circular Bootstrap shape: {bs_circular.shape}")

In [ ]:
# Compare distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

single_returns.hist(bins=50, ax=axes[0, 0], alpha=0.7)
axes[0, 0].set_title('Original Returns')

bs_iid.mean().hist(bins=50, ax=axes[0, 1], alpha=0.7, color='orange')
axes[0, 1].set_title('IID Bootstrap')

bs_stationary.mean().hist(bins=50, ax=axes[1, 0], alpha=0.7, color='green')
axes[1, 0].set_title('Stationary Bootstrap')

bs_circular.mean().hist(bins=50, ax=axes[1, 1], alpha=0.7, color='red')
axes[1, 1].set_title('Circular Bootstrap')

plt.tight_layout()
plt.show()

## 10. Clustering & Regime Detection <a id='clustering'></a>

Asset clustering and market regime analysis.

In [ ]:
from myPortfolioManagement.myClustering import (
    ts_clustering,
    cluster_ftca,
    detect_regimes
)

In [ ]:
# Time Series Clustering (DTW)
print("Running time series clustering...")
try:
    clusters, cluster_centers = ts_clustering(
        df=prices,
        number_of_clusters=3,
        algo='dtw',
        plot_bar_center=False
    )
    print("\nAsset Clusters:")
    print(clusters)
    print(f"\nCluster Centers shape: {cluster_centers.shape}")
except Exception as e:
    print(f"Time series clustering: {e}")

In [ ]:
# FTCA Clustering
print("Running FTCA clustering...")
try:
    ftca_clusters = cluster_ftca(returns_daily, threshold=0.50, col_name='asset')
    print("\nFTCA Clusters:")
    print(ftca_clusters)
except Exception as e:
    print(f"FTCA clustering: {e}")

In [ ]:
# Regime Detection
print("Detecting market regimes...")
try:
    # Detect regimes in volatility proxy with more robust handling
    vol_series = returns_daily.iloc[:, 0].rolling(20).std() * np.sqrt(252)
    
    # Remove NaN values and ensure we have enough data
    vol_clean = vol_series.dropna()
    
    if len(vol_clean) < 10:
        print("Insufficient data for regime detection after removing NaNs")
    else:
        vol_df = pd.DataFrame(vol_clean.values, 
                             index=vol_clean.index, 
                             columns=['Volatility'])
        
        regimes = detect_regimes(
            df=vol_df,
            series='Volatility',
            optimal_clusters=2,  # Reduced from 3 for more stability
            metric='euclidean',  # Changed from 'dtw' - more robust
            plot_regimes=False
        )
        print("\nDetected Regimes:")
        print(regimes['Volatility_Regime'].value_counts())
        print("\nRegime Statistics:")
        print(vol_df.groupby(regimes['Volatility_Regime'])['Volatility'].agg(['mean', 'std', 'count']))
except Exception as e:
    print(f"Regime detection: {e}")
    print("Proceeding without regime analysis...")

## 11. Visualization <a id='plotting'></a>

Create insightful visualizations.

In [ ]:
from myPortfolioManagement.myPlots import (
    correlation_matrix,
    monthly_heatmap,
    scatter_plot_simple
)

In [ ]:
# Correlation Matrix
print("Generating correlation matrix...")
correlation_matrix(
    returns_daily,
    corr_limit=None,
    figsize=(12, 8),
    diagonal=True
)
plt.show()

In [ ]:
# Monthly Returns Heatmap
print("Generating monthly heatmap...")
fig = monthly_heatmap(
    returns=portfolio_returns,
    annot_size=8,
    figsize=(12, 8),
    cbar=True,
    eoy=True,
    show=True
)

In [ ]:
# Risk-Return Scatter Plot
risk_return = pd.DataFrame({
    'Return': returns_daily.mean() * 252,
    'Volatility': returns_daily.std() * np.sqrt(252)
})

scatter_plot_simple(risk_return, x='Volatility', y='Return')
plt.title('Risk-Return Profile')
plt.show()

## 12. Portfolio Selection <a id='portfolio-selection'></a>

Generate all possible portfolio combinations.

In [ ]:
from myPortfolioManagement.myPortfolioSelection import (
    possible_combinations
)

assets = ['SPY', 'QQQ', 'IWM', 'AGG', 'GLD']

In [ ]:
# All combinations of 2-4 assets
combinations = possible_combinations(
    assets_to_consider=assets,
    min_assets=2,
    max_assets=4
)

print(f"Number of possible portfolios (2-4 assets): {len(combinations)}")
print("\nFirst 10 combinations:")
for i, combo in enumerate(combinations[:10]):
    print(f"  {i+1}. {combo}")

In [ ]:
# Combinations with must-have assets
combinations_must_have = possible_combinations(
    assets_to_consider=assets,
    min_assets=3,
    max_assets=4,
    must_have=['SPY', 'AGG']
)

print(f"Portfolios that must include SPY and AGG: {len(combinations_must_have)}")
for combo in combinations_must_have:
    print(f"  {combo}")

## 13. Complete Workflow Example <a id='workflow'></a>

End-to-end portfolio management workflow.

In [ ]:
print("="*80)
print("COMPLETE WORKFLOW - END TO END EXAMPLE")
print("="*80)

print("""
This section demonstrates a complete portfolio management workflow:
1. Fetch data
2. Calculate returns
3. Optimize portfolio
4. Backtest strategy
5. Analyze performance
""")

In [ ]:
# Step 1: Data already fetched above
print("Step 1: Using previously fetched data...")
print(f"  Assets: {list(prices.columns)}")
print(f"  Period: {prices.index[0]} to {prices.index[-1]}")

In [ ]:
# Step 2: Split into training and testing
split_date = '2023-01-01'
returns_train = returns_daily[returns_daily.index < split_date]
returns_test = returns_daily[returns_daily.index >= split_date]

print(f"\nStep 2: Train/Test Split at {split_date}")
print(f"  Training: {len(returns_train)} days")
print(f"  Testing: {len(returns_test)} days")

In [ ]:
# Step 3: Optimize portfolios on training data
print("\nStep 3: Optimizing portfolios on training data...")

weights_hrp_train = HRP(
    model='HRP',
    returns_training=returns_train.fillna(0),
    weight_max=0.25,
    weight_min=0.02
)
print("  HRP weights optimized")

weights_equal_train = equal_weight_portfolio(returns_train)
print("  Equal weight portfolio created")

In [ ]:
# Step 4: Calculate out-of-sample returns
print("\nStep 4: Calculating out-of-sample returns...")

# HRP portfolio returns
hrp_weights_dict = weights_hrp_train['port_weight'].to_dict()
portfolio_hrp_test = (returns_test * pd.Series(hrp_weights_dict)).sum(axis=1)
portfolio_hrp_test.name = 'HRP'

# Equal weight returns
equal_weights_dict = weights_equal_train['port_naive'].to_dict()
portfolio_equal_test = (returns_test * pd.Series(equal_weights_dict)).sum(axis=1)
portfolio_equal_test.name = 'Equal_Weight'

# Combine for comparison
portfolio_comparison = pd.DataFrame({
    'HRP': portfolio_hrp_test,
    'Equal_Weight': portfolio_equal_test,
    'SPY_Benchmark': returns_test.iloc[:, 0]
})

print("  Out-of-sample portfolios calculated")

In [ ]:
# Step 5: Performance analysis
print("\nStep 5: Out-of-Sample Performance Analysis...")

test_stats = get_main_stats(portfolio_comparison, rf=0.04)
print("\nOut-of-Sample Performance Metrics:")
print(test_stats.round(4))

In [ ]:
# Cumulative returns
cum_returns = (1 + portfolio_comparison).cumprod()
print("\nCumulative Returns (End of Period):")
print(cum_returns.iloc[-1].round(4))

In [ ]:
# Visualize cumulative returns
fig, ax = plt.subplots(figsize=(14, 7))
cum_returns.plot(ax=ax, linewidth=2)
ax.set_title('Out-of-Sample Cumulative Returns', fontsize=14)
ax.set_ylabel('Growth of $1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the complete functionality of the myPortfolioManagement library:

### Key Findings
- Analyzed **{len(tickers)}** diverse assets
- Generated **7+** different portfolio strategies
- Tested multiple optimization methods (HRP, HERC, GMV, Max Sharpe, CVaR)
- Performed comprehensive backtesting and simulation

### Next Steps
1. Review portfolio weights and adjust constraints
2. Run more extensive backtests with different parameters
3. Implement rebalancing strategies
4. Add transaction costs and slippage
5. Consider regime-based allocation

### Resources
- Full documentation: See README.md
- Example scripts: examples_portfolio_management.py
- GitHub: [QuantitativePortfolioManagement](https://github.com/ferhat00/QuantitativePortfolioManagement)